# Анализ розничных цен

Пятёрочка и Магнит · Москва и Санкт-Петербург. Тестовые выгрузки подходят для проверки кода. Охват полной выборки проверяется отдельно.

In [ ]:
from pathlib import Path
from src.price_analysis import load_prices, daily_panel, price_changes, attach_product_links

LOCAL = Path('data/input/normalized')
ROOT = LOCAL if LOCAL.is_dir() else Path('../hse-economics-4year-thesis-retail-price-collector-main/data/normalized')
prices, files = load_prices(ROOT)
files.groupby(files['source'].str.split('/').str[1])['rows'].agg(['size', 'sum'])

In [ ]:
prices.groupby('chain').agg(stores=('storeId', 'nunique'), categories=('categoryId', 'nunique'), first_day=('logicalDate', 'min'), last_day=('logicalDate', 'max'))

## Дневная панель

Берём последнее наблюдение за логический день. Сравниваем соседние календарные дни. Пропуски остаются пропусками. Порог 1% перенесён из курсовой как настраиваемый параметр.

In [ ]:
panel = price_changes(daily_panel(prices), threshold=0.01)
panel.groupby('chain')[['changedRegular', 'changedEffective']].agg(['count', 'mean'])

In [ ]:
links = Path('data/input/product_links.csv')
if links.exists():
    matched_panel = attach_product_links(panel, links)
    display(matched_panel.groupby('chain')['canonicalProductId'].agg(['count', 'size']))

In [ ]:
from src.reporting import save_reports

save_reports(prices, panel, Path('output'))